In [ ]:
# %% [markdown]
# # EDA for Fake News Detection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_isot_dataset, load_kaggle_fake_news, load_liar_dataset, merge_datasets
from pathlib import Path

In [ ]:
raw_dir = "data/raw"

# Load ISOT
isot_df = load_isot_dataset(raw_dir)
print("ISOT shape:", isot_df.shape)
print("ISOT label distribution:\n", isot_df["label"].value_counts())

# Load Kaggle (optional)
kaggle_df = None
kaggle_raw = Path(raw_dir) / "kaggle"
if kaggle_raw.exists():
    kaggle_df = load_kaggle_fake_news(kaggle_raw)
    print("Kaggle shape:", kaggle_df.shape)
    print("Kaggle label distribution:\n", kaggle_df["label"].value_counts())

# Load LIAR (optional)
liar_df = None
liar_raw = Path(raw_dir) / "liar"
if (Path(raw_dir) / "liar").exists():
    liar_df = load_liar_dataset(raw_dir)
    print("LIAR shape:", liar_df.shape)
    print("LIAR label distribution:\n", liar_df["label"].value_counts())

In [ ]:
df = merge_datasets(isot_df, kaggle_df, liar_df).reset_index(drop=True)
print("Total merged dataset shape:", df.shape)
print("Global label distribution:\n", df["label"].value_counts())

# Plot label distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label")
plt.title("Label Distribution (0: Real, 1: Fake)")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()

In [ ]:
df["text_len"] = df["content"].str.len()
df["word_count"] = df["content"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df, x="text_len", hue="label", bins=50, ax=axes[0])
axes[0].set_title("Character length distribution (by label)")

sns.histplot(df, x="word_count", hue="label", bins=50, ax=axes[1])
axes[1].set_title("Word count distribution (by label)")

plt.tight_layout()
plt.show()

print("\nText length stats by label:")
print(df.groupby("label")["text_len"].describe())

In [ ]:
#!pip install wordcloud

from wordcloud import WordCloud

def plot_wordcloud(text_series, title):
    text = " ".join(text_series.fillna("").tolist())
    wc = WordCloud(
        width=400,
        height=300,
        background_color="white",
        max_words=100,
    ).generate(text)
    plt.figure(figsize=(6, 4))
    plt.imshow(wc, interpolation="bilinear")
    plt.title(title)
    plt.axis("off")
    plt.show()

plot_wordcloud(df[df["label"] == 0]["content"], "Real News")
plot_wordcloud(df[df["label"] == 1]["content"], "Fake News")

In [ ]:
from src.data_prep import preprocess_and_split

procd_dir = Path("data/processed")
preprocess_and_split(
    raw_dir="data/raw",
    processed_dir=str(procd_dir),
    clean_stopwords=True,
)

print("Processed CSVs saved at:", procd_dir)